# Assignment 4 continual-learning experiments
This notebook installs the package, validates the frozen dataset, runs selected experiments with checkpoint recovery, and exports results and figures. Do not run the full schedule until the protocol and dataset hashes are confirmed.

In [ ]:
!python -m pip install -q -e '.[experiment,reporting]'


In [ ]:
import json, platform, subprocess, sys
import torch
environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(environment, indent=2))


## Google Drive and dataset
Mount Drive, then set the package path to the frozen dataset ZIP or place the CSV in `data/processed/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATASET_PACKAGE = '/content/drive/MyDrive/rmt/assignment4_real_dataset_package.zip'
RESULTS_DIR = '/content/drive/MyDrive/rmt/results/raw'
CHECKPOINT_DIR = '/content/drive/MyDrive/rmt/checkpoints'


In [ ]:
!python scripts/validate_dataset.py --dataset "$DATASET_PACKAGE" --output /content/dataset_validation.json


## Inspect the 19-run schedule

In [ ]:
from rmt.training.runner import create_run_schedule
schedule = create_run_schedule()
[(run.run_id, run.mode, run.parameters) for run in schedule]


## Run one experiment
This command initializes the real model and starts one planned run. It writes the result immediately and skips an existing file on a later execution.

In [ ]:
RUN_ID = 'sequential_lora_seed0'
!python scripts/run_experiment.py --run-id "$RUN_ID" --output "$RESULTS_DIR"


## Resume the full schedule
Run this cell only for the approved final experiment. Existing run files are skipped.

In [ ]:
# !python scripts/run_experiment.py --full --output "$RESULTS_DIR"


## Export tables and figures

In [ ]:
!python scripts/analyze_results.py --input "$RESULTS_DIR" --output /content/results/tables/run_metrics.csv
!python scripts/generate_figures.py --input /content/results/tables --output /content/results/figures
!cp -r /content/results /content/drive/MyDrive/rmt/results_export
